# 03 · Classifier arah (kurs naik/turun besok), out-of-fold
Regresi logistik L2, TimeSeriesSplit (gap 5). Test tidak dinilai di sini.

In [1]:
import json
import numpy as np, pandas as pd
from utils import PROC, load, oof_pup, auc_ci

df, price = load(with_pup=False)
text_cols = [c for c in df.columns if c not in price + ['target', 'target2', 'target0', 'split']]
y = (df.target > 0).astype(int)
tr, va = df.split == 'train', df.split == 'val'
print(len(text_cols), 'fitur teks | Train naik (mayoritas):', y[tr].mean().round(3))

57 fitur teks | Train naik (mayoritas): 0.592


In [2]:
p_up = oof_pup(df, text_cols)

m = tr & p_up.notna()                      # hari fold pertama tidak punya OOF → dibuang
yy, pp = y[m].values, p_up[m].values
auc, (lo, hi) = auc_ci(yy, pp)
maj = max(yy.mean(), 1 - yy.mean())
gate = bool(lo > 0.5)
print(f'OOF Train n={m.sum()}: AUC={auc:.3f} [{lo:.3f}, {hi:.3f}] | akurasi={((pp > .5) == yy).mean():.3f} vs mayoritas={maj:.3f}')
print('Val AUC:', round(auc_ci(y[va].values, p_up[va].values)[0], 3))
print('GATE lolos (batas bawah > 0.5):', gate)
if not gate: print('→ classifier dianggap tidak punya sinyal. Hasil ini sah dilaporkan; varian D tetap dihitung sebagai pembanding.')

OOF Train n=80: AUC=0.645 [0.500, 0.780] | akurasi=0.575 vs mayoritas=0.613
Val AUC: 0.509
GATE lolos (batas bawah > 0.5): True


In [3]:
p_up.rename_axis('date').reset_index().to_parquet(PROC / 'pup.parquet')
json.dump(dict(gate=gate, auc_oof=auc, ci=[lo, hi], n_oof=int(m.sum())), open(PROC / 'gate.json', 'w'))